In [1]:
source("utils.R")
suppressPackageStartupMessages({
  library(glue)
  library(S4Vectors)
  library(data.table)
  library(IRanges)
  library(GenomicRanges)
  library(GenomeInfoDb)
  library(GenomicFeatures)
})

## cCRE
#### Step 1: Download from ENCODE SCREEN (cell-type agnostic cCREs)  
hg38: https://downloads.wenglab.org/Registry-V4/GRCh38-cCREs.bed  
mm10: https://downloads.wenglab.org/Registry-V4/mm10-cCREs.bed


#### Step 2: Convert bed to GRanges 
hg38: 2,348,854 cCREs  
mm10: 926,843 cCREs

In [2]:
CCRE_EXTRA_COLS <- c("cCRE_id", "ccre_class")

out_files <- c('GRCh38-cCREs.bed', 'mm10-cCREs.bed')

for (out_file in out_files) {

  ref <- if (grepl("GRCh38|hg38", out_file)) {
    "hg38"
  } else if (grepl("mm10", out_file)) {
    "mm10"
  } else {
    stop("Unknown genome in bed name: ", out_file)
  }
  cat(sprintf("\n==================== %s ====================\n", ref))

  gr <- bed_to_gr(out_file, extra_cols = CCRE_EXTRA_COLS)
  gr$ccre_group <- sub(",.*$", "", gr$ccre_class)
  GenomeInfoDb::genome(gr) <- ref

  class_tab <- sort(table(gr$ccre_class), decreasing = TRUE)
  print(cbind(n = class_tab, pct = round(100 * prop.table(class_tab), 1)))
  message()

  out_path <- sprintf("ENCODE_cCRE_%s.rds", ref)
  saveRDS(gr, file = out_path, compress = "xz")
}


==================== hg38 ====================
                 n  pct
dELS       1469205 62.5
pELS        249464 10.6
CA          245985 10.5
CA-CTCF     126034  5.4
TF          105286  4.5
CA-H3K4me3   79246  3.4
PLS          47532  2.0
CA-TF        26102  1.1



==================== mm10 ====================
                n  pct
dELS       400783 43.2
CA         291800 31.5
pELS       111218 12.0
CA-CTCF     45933  5.0
PLS         27332  2.9
CA-H3K4me3  23832  2.6
TF          15238  1.6
CA-TF       10707  1.2


## RepeatMasker
#### Step 1: Download from RepeatMasker website

hg38: https://www.repeatmasker.org/genomes/hg38/rmsk4.0.5_rb20140131/hg38.fa.out.gz  
mm10: https://www.repeatmasker.org/genomes/mm10/rmsk4.0.5_rb20140131/mm10.fa.out.gz  

#### Step 2: Convert to GRanges

The raw RepeatMasker output (`*.fa.out.gz`) is a whitespace-aligned text table, not a BED file. It starts with a 3-line header (two title lines and a blank line), followed by one row per repeat annotation:

```
   SW  perc perc perc  query      position in query           matching       repeat              position in  repeat
score  div. del. ins.  sequence    begin     end    (left)    repeat         class/family         begin  end (left)   ID

  463   1.3  0.6  1.7  chr1        10001    10468 (248945954) +  (TAACCC)n      Simple_repeat            1  471    (0)      1
 3612  11.4 21.5  1.3  chr1        11678    12044 (248944378) C  L1MC5a         LINE/L1              (2112) 5810   5434     2  *
```

| Col | Field | Description |
|---|---|---|
| 1 | SW score | Smith-Waterman alignment score; higher means more similar to the repeat consensus |
| 2 | perc div | % divergence from the consensus; a rough proxy for element age (older insertions are more diverged) |
| 3 | perc del | % deletions relative to the consensus |
| 4 | perc ins | % insertions relative to the consensus |
| 5 | query sequence | Chromosome / contig |
| 6–7 | begin / end | Genomic coordinates, **1-based, closed interval** |
| 8 | (left) | Bases remaining on the chromosome after the match |
| 9 | strand | `+` (forward) or `C` (complement, i.e. minus strand) |
| 10 | matching repeat | Repeat name, e.g. `AluY`, `L1HS`, `(TAACCC)n` |
| 11 | class/family | e.g. `SINE/Alu`, `LINE/L1`, `LTR/ERVK`; some have no family (`Simple_repeat`); `?` marks uncertain assignments (`LTR?`, `DNA/hAT?`) |
| 12–14 | position in repeat | Matched region within the consensus; order is `begin, end, (left)` for `+` and `(left), end, begin` for `C` |
| 15 | ID | Fragments sharing an ID are assigned to the same insertion event (e.g. an L1 interrupted by another element) |
| 16 | `*` (optional) | The region overlaps another annotation with a higher score |


In [3]:
rm_out_to_gr <- function(out_file) {
  all_lines <- readLines(out_file, warn = FALSE)
  all_lines <- all_lines[nzchar(trimws(all_lines))]
  data_start_idx <- which(!grepl("^[A-Za-z]", trimws(all_lines)))[1]
  data_lines <- trimws(all_lines[data_start_idx:length(all_lines)])

  mat <- stringr::str_split(data_lines, "\\s+", simplify = TRUE)
  df <- as.data.frame(mat, stringsAsFactors = FALSE)
  colnames(df) <- paste0("X", seq_len(ncol(df)))
  df$strand <- ifelse(df$X9 == "C", "-", "+")

  gr <- makeGRangesFromDataFrame(
    df, keep.extra.columns = FALSE,
    seqnames.field = "X5", start.field = "X6", end.field = "X7", strand.field = "strand"
  )
  mcols(gr) <- S4Vectors::DataFrame(
    score     = as.integer(df$X1),
    perc_div  = as.numeric(df$X2),
    perc_del  = as.numeric(df$X3),
    perc_ins  = as.numeric(df$X4),
    rep_name  = df$X10,
    rep_class = df$X11
  )

  cat(sprintf("  Parsed %d records on %d chromosomes/contigs\n",
              length(gr), length(unique(seqnames(gr)))))
  cat("  "); check_overlap_self(gr)
  gr
}

#### Step 3: Deal with overlap
In the raw RepeatMasker output, the same genomic region can be covered by more than one annotation. This matters for any per-base analysis, e.g. "which repeat class does this peak fall in?" or "how many bp of the genome are LINEs?". With overlaps, a base can be counted in several classes, and summed lengths exceed the actual genomic coverage.  
  
To resolve this, the annotations are split into non-overlapping segments, and each segment is assigned a single representative repeat:

```
Original:   A [======= AluY, score 2000 =======]
            B                [====== L1MC, score 800 ======]

Disjoin:      [   p1        ][       p2        ][    p3     ]
              from {A}        from {A, B}         from {B}

Select:       p1 = AluY       p2 = AluY           p3 = L1MC
                              (higher score wins)

Merge:        [=========== AluY ===============][   L1MC    ]
              (adjacent pieces with the same repeat name are joined)
```

When several annotations compete for a segment, the representative is chosen by:

1. highest alignment `score`,
2. then lowest `perc_div`,
3. then the largest genomic span.

In [4]:
rm_make_nonoverlap <- function(gr) {
  pieces <- disjoin(gr, with.revmap = TRUE)
  rev <- mcols(pieces)$revmap
  cat(sprintf("  Disjoin: %d records -> %d pieces (%d need selection)\n",
              length(gr), length(pieces), sum(lengths(rev) > 1)))

  hits <- data.table(
    piece = rep(seq_along(rev), lengths(rev)),
    idx   = unlist(rev, use.names = FALSE)
  )
  hits[, `:=`(
    score = gr$score[idx],
    div   = gr$perc_div[idx],
    w     = width(gr)[idx]
  )]
  setorder(hits, piece, -score, div, -w)
  picked <- hits[!duplicated(piece), idx]

  pieces <- granges(pieces)
  pieces$rep_name  <- gr$rep_name[picked]
  pieces$rep_class <- gr$rep_class[picked]

  pieces <- sort(pieces)
  n  <- length(pieces)
  sn <- as.character(seqnames(pieces))
  st <- as.character(strand(pieces))
  nm <- pieces$rep_name
  can_merge <- sn[-n] == sn[-1] & st[-n] == st[-1] &
               end(pieces)[-n] + 1L == start(pieces)[-1] & nm[-n] == nm[-1]

  first_idx <- which(c(TRUE, !can_merge))
  last_idx  <- c(first_idx[-1] - 1L, n)
  out <- GRanges(
    seqnames  = seqnames(pieces)[first_idx],
    ranges    = IRanges(start(pieces)[first_idx], end(pieces)[last_idx]),
    strand    = strand(pieces)[first_idx],
    rep_name  = nm[first_idx],
    rep_class = pieces$rep_class[first_idx]
  )

  cat(sprintf("  Merge:   %d adjacent same-name pairs -> %d regions\n",
              sum(can_merge), length(out)))
  cat("  "); check_overlap_self(out)
  out
}

#### Step 4: Class Filtering
RepeatMasker labels repeats with a hierarchical `class/family` string (e.g. `LINE/L1`, `SINE/Alu`, `LTR/ERVK`). Some labels also carry `?` to mark an uncertain assignment (e.g. `LTR?`, `DNA/hAT?`). In total there are ~60+ distinct labels, many of which cover only a small number of regions.

Downstream analyses (e.g. overlap/enrichment of peaks across repeat types) work better with a small set of broad, well-populated categories. Fine-grained families split the data into many tiny groups, which gives noisy and hard-to-interpret statistics. The labels are therefore simplified in two steps:

1. **Keep only the main class.** The family part after `/` is dropped, e.g. `LINE/L1` → `LINE`, `SINE/Alu` → `SINE`, `LTR/ERVL-MaLR` → `LTR`.
2. **Reassign uncertain main classes to `Unknown`.** If the main class itself contains `?`, the region is relabeled `Unknown` instead of being forced into a class it may not belong to. Examples: `LTR?` → `Unknown`, `DNA?` → `Unknown`, `RC?/Helitron?` → `Unknown`, `SINE?/tRNA` → `Unknown`.

After filtering, the ~60+ original labels collapse into about 15 main classes. These are the major transposable element classes (`LINE`, `SINE`, `LTR`, `DNA`, `Retroposon`, `RC`), the non-TE repeats (`Simple_repeat`, `Low_complexity`, `Satellite`), the small RNA-derived repeats (`snRNA`, `tRNA`, `rRNA`, `srpRNA`, `scRNA`, `RNA`), and `Unknown`.

In [5]:
rm_filter_main_classes <- function(gr) {
  main_class <- sub("/.*", "", gr$rep_class)
  has_question <- grepl("\\?", main_class)
  main_class[has_question] <- "Unknown"

  cat(sprintf("  Labels: %d -> %d\n",
              length(unique(gr$rep_class)), length(unique(main_class))))
  cat(sprintf("  Reassigned to 'Unknown': %d regions (%.2f%%)\n\n",
              sum(has_question), 100 * mean(has_question)))

  gr$rep_class <- main_class
  class_tab <- sort(table(gr$rep_class), decreasing = TRUE)
  print(cbind(n = class_tab, pct = round(100 * prop.table(class_tab), 1)))
  gr
}

In [6]:
out_files <- c("hg38.fa.out.gz", "mm10.fa.out.gz")

for (out_file in out_files) {
  ref <- if (grepl("hg38", out_file)) {
    "hg38"
  } else if (grepl("mm10", out_file)) {
    "mm10"
  } else {
    stop("Unknown genome in out_file: ", out_file)
  }

  cat(sprintf("\n==================== %s ====================\n", ref))

  cat("\n[1/3] Parse RepeatMasker .out\n")
  gr <- rm_out_to_gr(out_file)

  cat("\n[2/3] Remove overlaps\n")
  gr <- rm_make_nonoverlap(gr)

  cat("\n[3/3] Simplify classes\n")
  gr <- rm_filter_main_classes(gr)

  GenomeInfoDb::genome(gr) <- ref
  out_path <- sprintf("RepeatMasker_%s.rds", ref)
  saveRDS(gr, file = out_path, compress = "xz")
  cat(sprintf("\nSaved: %s\n", out_path))
}


==================== hg38 ====================

[1/3] Parse RepeatMasker .out
  Parsed 5622516 records on 455 chromosomes/contigs
  Found 248856 internal overlapping interval pairs

[2/3] Remove overlaps
  Disjoin: 5622516 records -> 5745451 pieces (124427 need selection)
  Merge:   180640 adjacent same-name pairs -> 5564811 regions
  No internal overlaps found

[3/3] Simplify classes
  Labels: 66 -> 16
  Reassigned to 'Unknown': 13303 regions (0.24%)

                     n  pct
SINE           1855992 33.4
LINE           1580250 28.4
LTR             747024 13.4
Simple_repeat   700280 12.6
DNA             530285  9.5
Low_complexity  103235  1.9
Unknown          19821  0.4
Satellite         7887  0.1
Retroposon        5699  0.1
snRNA             4534  0.1
tRNA              2065  0.0
scRNA             1929  0.0
rRNA              1834  0.0
RC                1811  0.0
srpRNA            1404  0.0
RNA                761  0.0

Saved: RepeatMasker_hg38.rds

==================== mm10 =========

### ChromHMM

#### Step 1: Download from enrstlab website
hg38: https://public.hoffman2.idre.ucla.edu/ernst/2K9RS//full_stack/full_stack_annotation_public_release/hg38/hg38_genome_100_segments.bed.gz  
mm10: https://public.hoffman2.idre.ucla.edu/ernst/2K9RS//full_stack/full_stack_annotation_public_release/mm10/mm10_100_segments_segments.bed.gz  

#### Step 2: Convert to GRanges \& Filter states

The full-stack ChromHMM annotations (Ernst lab) are universal chromatin state maps: a single 100-state model trained jointly on many cell and tissue types. The resulting annotation is the same for all cell types; it is not specific to any one of them.

- **hg38**: trained on 1,032 datasets from 127 reference epigenomes. The model was trained on hg19, and the hg38 annotation was produced by liftOver.
- **mm10**: trained on 901 datasets assaying 14 chromatin marks across 26 cell or tissue types.

The `*_segments.bed.gz` files are 4-column BED files:
```
chr1    0        3000000  1_mGapArtf1
chr1    3000200  3001400  57_mEnhA1
...
```

State labels follow the pattern `<index>_<prefix><group><sub-state>`:

| Example | Index | Prefix | Group | Sub-state |
|---|---|---|---|---|
| `57_EnhA1` (human) | `57` | – | `EnhA` | `1` |
| `57_mEnhA1` (mouse) | `57` | `m` | `EnhA` | `1` |

- **Index** (1–100) orders the states.
- **Prefix** `m` appears only in mouse labels, to distinguish them from human states.
- **Group** is the biological category, e.g. `TSS`, `PromF` (promoter flank), `BivProm` (bivalent promoter), `EnhA` / `EnhWk` (active / weak enhancer), `TxEnh` (transcribed enhancer), `Tx` / `TxEx` / `TxWk` (transcription), `ReprPC` (polycomb repressed), `Het` (heterochromatin), `ZNF` (zinc finger genes), `Quies` (quiescent), `GapArtf` (assembly gaps and artifacts).
- **Sub-state** distinguishes states within the same group.

To make this annotation easier to use downstream, the raw segments are processed in three steps:  
1. **Read the 4th column** as the state label (`state`). The original label is kept so that analyses at the level of all 100 states remain possible.
2. **Collapse states into groups.** The leading index (`57_`), the mouse prefix (`m`) and the trailing sub-state number (`1`) are removed, e.g. `57_mEnhA1` → `EnhA` and `12_EnhA1` → `EnhA`. Downstream analyses (e.g. peak overlap and enrichment) are easier to interpret with a small number of well-populated groups than with 100 fine-grained states.
3. **Remove `GapArtf`.** These segments correspond to assembly gaps and artifacts rather than real chromatin states. They span large stretches of the genome, so keeping them would distort genome-length–based proportions and enrichment statistics.


In [7]:
chmm_out_to_gr <- function(out_file){
  gr <- bed_to_gr(out_file)
  colnames(mcols(gr)) <- "state"
  cat(sprintf("Parsed %d segments, %d states\n", length(gr), length(unique(gr$state))))
  gr
}

chmm_filter_state <- function(gr){
  gr$clean_class <- sub("\\d+$", "", sub("^\\d+_m?", "", gr$state))
  class_tab <- sort(table(gr$clean_class), decreasing = TRUE)
  print(cbind(n = class_tab, pct = round(100 * prop.table(class_tab), 1)))

  is_gap <- gr$clean_class == "GapArtf"
  cat(sprintf("Removed GapArtf: %d segments (%.1f Mb)\n",
              sum(is_gap), sum(as.numeric(width(gr)[is_gap])) / 1e6))
  gr[!is_gap]
}

In [8]:
out_files <- c("hg38_genome_100_segments.bed.gz", "mm10_100_segments_segments.bed.gz")

for (out_file in out_files){
  ref <- if (grepl("hg38", out_file)) {
    "hg38"
  } else if (grepl("mm10", out_file)) {
    "mm10"
  } else {
    stop("Unknown genome in out_file: ", out_file)
  }
  cat(sprintf("\n==================== %s ====================\n", ref))

  gr <- chmm_out_to_gr(out_file)
  gr <- chmm_filter_state(gr)

  GenomeInfoDb::genome(gr) <- ref
  out_path <- sprintf("ChromHMM_%s.rds", ref)
  saveRDS(gr, file = out_path, compress = "xz")
  cat(sprintf("\nSaved: %s\n", out_path))
}


==================== hg38 ====================
Parsed 6098430 segments, 100 states
              n  pct
Quies   1586228 26.0
ReprPC   649638 10.7
Acet     648814 10.6
EnhA     633446 10.4
Tx       576422  9.5
HET      563529  9.2
EnhWk    560185  9.2
TxWk     265326  4.4
TxEnh    193238  3.2
TxEx     123883  2.0
PromF     91244  1.5
GapArtf   73075  1.2
BivProm   50008  0.8
znf       35469  0.6
TSS       26196  0.4
DNase     21729  0.4
Removed GapArtf: 73075 segments (140.4 Mb)

Saved: ChromHMM_hg38.rds

==================== mm10 ====================
Parsed 5929546 segments, 100 states
                   n  pct
HET          1205320 20.3
Quies        1194707 20.1
EnhWk         747058 12.6
Tx            505407  8.5
ReprPC        466345  7.9
EnhA          444251  7.5
OpenC         443986  7.5
TxWk          251552  4.2
GapArtf       238425  4.0
TxEnh         170865  2.9
PromF          77480  1.3
ReprPC_openC   67212  1.1
TxEx           46090  0.8
TSS            25974  0.4
ZNF            2

## GENCODE Gene Annotation
#### Step 1: Download from GENCODE website
hg38: https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_49/gencode.v49.basic.annotation.gtf.gz  
mm10: https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_mouse/release_M25/gencode.vM25.basic.annotation.gtf.gz  

#### Step 2: Convert to RGanges \& Gene-level filtering
Before selecting representative transcripts, genes are filtered to exclude categories that typically lack RNA transcripts. The following gene types are removed:

- **Pseudogenes**: `processed_pseudogene`, `unprocessed_pseudogene`, `unitary_pseudogene`, `rRNA_pseudogene`
- **Immunoglobulin and T-cell receptor pseudogenes**: `IG_V_pseudogene`, `IG_J_pseudogene`, `IG_C_pseudogene`, `TR_V_pseudogene`, `TR_J_pseudogene`, `IG_pseudogene`
- **Artifacts**: `artifact`

After this filtering step:
- **hg38**: 66,413 genes remain (from 78,691 original genes)
- **mm10**: 42,396 genes remain (from 55,335 original genes)

In [9]:
EXCLUDE_GENE_TYPES <- c(
  "processed_pseudogene", "unprocessed_pseudogene", "unitary_pseudogene", "rRNA_pseudogene",
  "IG_V_pseudogene", "IG_J_pseudogene", "IG_C_pseudogene",
  "TR_V_pseudogene", "TR_J_pseudogene", "IG_pseudogene",
  "artifact"
)

gtf_to_gr <- function(gtf_file) {
  gr <- rtracklayer::import(gtf_file)
  cat(sprintf("  Imported %d features: %d genes, %d transcripts\n",
              length(gr), sum(gr$type == "gene"), sum(gr$type == "transcript")))
  if (!is(gr$tag, "List")) {
    warning("'tag' was imported as plain character; transcripts with multiple tags may have lost some of them.")
  }
  gr
}

gtf_filter_genes <- function(gr, exclude_types = EXCLUDE_GENE_TYPES) {
  genes <- gr[gr$type == "gene"]
  drop <- genes$gene_type %in% exclude_types

  cat("  Removed gene types:\n")
  print(cbind(n = sort(table(genes$gene_type[drop]), decreasing = TRUE)))

  gr <- gr[!gr$gene_id %in% genes$gene_id[drop]]
  cat(sprintf("  Genes: %d -> %d\n", length(genes), sum(gr$type == "gene")))
  gr
}

#### Step 3: Transcript validation and evidence levels
In the original `basic.annotation.CHR` GTF files, each gene is associated with at least one transcript, and many genes have multiple transcripts.
- **hg38**: Among 66,413 filtered genes, 27,976 genes have more than one annotated transcript.   
- **mm10**: Among 42,396 filtered genes, 13,887 genes have more than one annotated transcript.   

Not all annotated transcripts are experimentally validated. GENCODE integrates evidence from multiple sources, and the presence of a transcript in the annotation does not necessarily imply direct experimental confirmation. Several annotation fields are commonly used to assess transcript reliability and representativeness:
- `tag ∈ {MANE_Select, appris_principal_1–4}`  
  These tags indicate that a transcript is designated as a representative or principal isoform under the MANE or APPRIS frameworks, and is often used as a standardized reference for functional or comparative analyses.
- `transcript_support_level`  
  This field reflects the degree of support from mRNA and EST alignment evidence. Lower values correspond to stronger transcriptional support.
- `havana_transcript`  
  This annotation indicates that the transcript originates from the HAVANA manual curation pipeline, implying human expert review and integration rather than purely automated prediction.

When a single representative transcript per gene is required, transcripts can be prioritized according to the following hierarchy:

1. MANE_Select  
2. appris_principal_1  
3. appris_principal_2  
4. appris_principal_3  
5. appris_principal_4  
6. Presence of a defined `transcript_support_level`  
7. Presence of `havana_transcript`

- **hg38**: 46,922 of 66,413 genes have at least one 'validated' transcript
- **mm10**: 42,393 of 42,396 genes have at least one 'validated' transcript

In [10]:
REP_LEVELS <- c("MANE_Select",
                "appris_principal_1", "appris_principal_2",
                "appris_principal_3", "appris_principal_4",
                "TSL", "havana", "none (longest)")

gtf_select_rep_transcript <- function(gr) {
  tx <- gr[gr$type == "transcript"]
  n_tx <- table(tx$gene_id)
  cat(sprintf("  Genes with >1 transcript: %d of %d\n", sum(n_tx > 1), length(n_tx)))

  tags <- if (is(tx$tag, "List")) S4Vectors::unstrsplit(tx$tag, ",") else as.character(tx$tag)
  tags[is.na(tags)] <- ""
  tsl <- suppressWarnings(as.integer(tx$transcript_support_level))   # "NA" -> NA

  rank <- rep(8L, length(tx)) 
  rank[!is.na(tx$havana_transcript)] <- 7L
  rank[!is.na(tsl)] <- 6L
  for (k in 4:1) {
    rank[grepl(sprintf("(^|,)appris_principal_%d(,|$)", k), tags)] <- k + 1L
  }
  rank[grepl("(^|,)MANE_Select(,|$)", tags)] <- 1L

  cat(sprintf("  Genes with >=1 validated transcript: %d of %d\n",
              length(unique(tx$gene_id[rank <= 7])), length(n_tx)))

  o <- order(tx$gene_id, rank, tsl, -width(tx), na.last = TRUE)
  first <- o[!duplicated(tx$gene_id[o])]
  rep_tx   <- tx[first]
  rep_rank <- rank[first]

  cat("  Representative transcript chosen by:\n")
  sel_tab <- table(factor(REP_LEVELS[rep_rank], levels = REP_LEVELS))
  print(cbind(n = sel_tab, pct = round(100 * prop.table(sel_tab), 1)))

  gr <- gr[gr$type == "gene" | gr$transcript_id %in% rep_tx$transcript_id]
  gr$rep_rank <- rep_rank[match(gr$gene_id, rep_tx$gene_id)]
  gr
}

### Step 4: Extract Representative Gene Bodies (TSS–TES)

For gene-level annotation, each gene is represented by a single genomic interval spanning the TSS to the TES of its representative transcript (selected in Step 3).

The `gene` records in the GTF are not used for this purpose: their coordinates are the union of all transcripts of the gene. A rarely used isoform that extends far upstream or downstream would shift the gene boundaries, and any TSS- or TES-based windows defined from them. Using the representative transcript places the TSS and TES at the gene's principal transcription start and end.

Only the `transcript` records are kept, with the following attributes:

| Column | Description |
|---|---|
| `gene_id` | GENCODE gene ID |
| `gene_name` | Gene symbol |
| `gene_type` | Gene biotype (e.g. `protein_coding`, `lncRNA`) |
| `transcript_id` | ID of the representative transcript |
| `rep_rank` | Evidence level used to select the representative transcript (1 = MANE_Select, …, 8 = no evidence, longest transcript) |

Strand is preserved, so the TSS is `start` on `+` and `end` on `−`. Promoter, gene body and TES-flanking regions are not precomputed here; they are derived at annotation time from user-defined window sizes.

In [11]:
gtf_extract_gene <- function(gr) {
  gene_gr <- gr[gr$type == "transcript"]
  stopifnot(!anyDuplicated(gene_gr$gene_id))

  mcols(gene_gr) <- mcols(gene_gr)[, c("gene_id", "gene_name", "gene_type", "transcript_id", "rep_rank")]
  cat(sprintf("  Genes: %d (one representative transcript each)\n", length(gene_gr)))

  type_tab <- sort(table(gene_gr$gene_type), decreasing = TRUE)
  print(cbind(n = type_tab, pct = round(100 * prop.table(type_tab), 1)))
  gene_gr
}

In [12]:
gtf_files <- list(
  hg38 = "gencode.v49.basic.annotation.gtf.gz",
  mm10 = "gencode.vM25.basic.annotation.gtf.gz"
)

for (ref in names(gtf_files)) {
  gtf_file <- gtf_files[[ref]]
  cat(sprintf("\n==================== %s ====================\n", ref))

  cat("\n== Convert to GRanges & gene-level filtering ==\n")
  gr <- gtf_to_gr(gtf_file)
  gr <- gtf_filter_genes(gr)

  cat("\n== Representative transcript selection ==\n")
  gr <- gtf_select_rep_transcript(gr)
  GenomeInfoDb::genome(gr) <- ref

  # Save for genic annotation
  rep_path <- sprintf("GENCODE_rep_%s.rds", ref)
  saveRDS(gr, file = rep_path, compress = "xz")
  cat(sprintf("\nSaved: %s\n", rep_path))

  cat("\n== Extract representative gene bodies ==\n")
  gene_gr <- gtf_extract_gene(gr)

  gene_path <- sprintf("GENCODE_gene_%s.rds", ref)
  saveRDS(gene_gr, file = gene_path, compress = "xz")
  cat(sprintf("\nSaved: %s\n", gene_path))
}


==================== hg38 ====================

== Convert to GRanges & gene-level filtering ==
  Imported 5868512 features: 78691 genes, 280000 transcripts


Warning message in gtf_to_gr(gtf_file):
"'tag' was imported as plain character; transcripts with multiple tags may have lost some of them."


  Removed gene types:
                          n
processed_pseudogene   9487
unprocessed_pseudogene 1949
rRNA_pseudogene         497
IG_V_pseudogene         187
unitary_pseudogene       89
TR_V_pseudogene          33
artifact                 19
IG_C_pseudogene           9
TR_J_pseudogene           4
IG_J_pseudogene           3
IG_pseudogene             1
  Genes: 78691 -> 66413

== Representative transcript selection ==
  Genes with >1 transcript: 27976 of 66413
  Genes with >=1 validated transcript: 39855 of 66413
  Representative transcript chosen by:
                       n  pct
MANE_Select           41  0.1
appris_principal_1   897  1.4
appris_principal_2    92  0.1
appris_principal_3    38  0.1
appris_principal_4   199  0.3
TSL                27711 41.7
havana             10877 16.4
none (longest)     26558 40.0

Saved: GENCODE_rep_hg38.rds

== Extract representative gene bodies ==
  Genes: 66413 (one representative transcript each)
                                       n  pct


Warning message in gtf_to_gr(gtf_file):
"'tag' was imported as plain character; transcripts with multiple tags may have lost some of them."


  Removed gene types:
                           n
processed_pseudogene   10002
unprocessed_pseudogene  2723
IG_V_pseudogene          158
unitary_pseudogene        61
TR_V_pseudogene           34
TR_J_pseudogene           10
IG_pseudogene              2
IG_C_pseudogene            1
  Genes: 55401 -> 42410

== Representative transcript selection ==
  Genes with >1 transcript: 13771 of 42410
  Genes with >=1 validated transcript: 36105 of 42410
  Representative transcript chosen by:
                       n  pct
MANE_Select            0  0.0
appris_principal_1  1877  4.4
appris_principal_2     0  0.0
appris_principal_3     0  0.0
appris_principal_4     0  0.0
TSL                27333 64.4
havana              6895 16.3
none (longest)      6305 14.9

Saved: GENCODE_rep_mm10.rds

== Extract representative gene bodies ==
  Genes: 42410 (one representative transcript each)
                                       n  pct
protein_coding                     21859 51.5
lincRNA                      

## GENCODE Genic Annotation
#### Step 1–3: Same as GENCODE Gene Annotation

The genic annotation starts from the same preprocessing as the gene annotation: download, gene-level filtering, and selection of one representative transcript per gene. Using a single representative transcript keeps each gene from contributing overlapping promoters, exons or introns from multiple isoforms. The full transcript structure (`transcript`, `exon`, `CDS`, `UTR`) saved after Step 3 is used as input for the steps below.


#### Step 4: Extract Promoter, Intron & UTR

The GENCODE GTF provides `gene`, `transcript`, `exon`, `CDS` and a generic `UTR` feature, but not promoters, introns, or separate 5'/3' UTRs, which are commonly used to annotate peaks. These regions are derived from the representative transcript of each gene:

- **Promoter**: the 1 kb immediately upstream of the transcription start site (TSS), excluding the TSS itself and defined according to strand (`[start − 1000, start − 1]` on `+`, `[end + 1, end + 1000]` on `−`).
- **Intron**: the gaps between consecutive exons of a transcript. Single-exon transcripts have no introns.
- **5' / 3' UTR**: GENCODE labels all untranslated exon segments as `UTR`. Each segment is assigned to the 5' or 3' UTR according to its position relative to the transcript's CDS, taking strand into account (on `+`, segments upstream of the CDS are 5' UTR and downstream segments are 3' UTR; the reverse on `−`).

Each derived region inherits the gene and transcript attributes of its parent transcript. The generic `UTR` feature and the short features `start_codon`, `stop_codon` and `Selenocysteine` are then removed.


In [13]:
gtf_add_regions <- function(gr, promoter_upstream = 1000L) {
  gr$type <- as.character(gr$type)
  tx  <- gr[gr$type == "transcript"]
  exn <- gr[gr$type == "exon"]
  cds <- gr[gr$type == "CDS"]
  utr <- gr[gr$type == "UTR"]

  # Promoter: the region spanning promoter_upstream bp upstream of the TSS, excluding the TSS and accounting for strand orientation
  prom <- promoters(granges(tx), upstream = promoter_upstream, downstream = 0)
  prom$transcript_id <- tx$transcript_id
  prom <- prom[end(prom) >= 1]
  start(prom) <- pmax(start(prom), 1L)
  prom$type <- "Promoter"

  # Intron: transcript - extron
  ebt <- split(granges(exn), exn$transcript_id)
  intr_list <- psetdiff(unlist(range(ebt)), ebt)
  intr <- unlist(intr_list, use.names = FALSE)
  intr$transcript_id <- rep(names(ebt), lengths(intr_list))
  intr$type <- "Intron"

  # UTR: split into 5' and 3'
  cds_rng <- unlist(range(split(granges(cds), cds$transcript_id)))
  i  <- match(utr$transcript_id, names(cds_rng))
  cs <- start(cds_rng)[i]
  ce <- end(cds_rng)[i]
  st <- as.character(strand(utr))
  is5 <- which((st == "+" & end(utr) < cs) | (st == "-" & start(utr) > ce))
  is3 <- which((st == "+" & start(utr) > ce) | (st == "-" & end(utr) < cs))
  utr_g <- granges(utr)
  utr_g$transcript_id <- utr$transcript_id
  utr5 <- utr_g[is5]; utr5$type <- "5' UTR"
  utr3 <- utr_g[is3]; utr3$type <- "3' UTR"

  cat(sprintf("  Added: %d promoters, %d introns, %d 5' UTRs, %d 3' UTRs\n",
              length(prom), length(intr), length(utr5), length(utr3)))
  n_unassigned <- length(utr) - length(is5) - length(is3)
  if (n_unassigned > 0) cat(sprintf("  UTR segments not assigned 5'/3': %d\n", n_unassigned))

  new <- c(prom, intr, utr5, utr3)
  meta_cols <- setdiff(colnames(mcols(gr)), c("type", "transcript_id"))
  idx <- match(new$transcript_id, tx$transcript_id)
  mcols(new)[, meta_cols] <- mcols(tx)[idx, meta_cols]
  mcols(new) <- mcols(new)[, colnames(mcols(gr))]

  # rename type names
  gr$type[gr$type == "exon"] <- "Exon"
  drop_types <- c("UTR", "Selenocysteine", "start_codon", "stop_codon")
  gr <- sort(c(gr[!gr$type %in% drop_types], new))

  type_tab <- sort(table(gr$type), decreasing = TRUE)
  print(cbind(n = type_tab, pct = round(100 * prop.table(type_tab), 1)))
  gr
}


### Step 5: Deal with Overlap

A genomic position can belong to several features at once. For example, CDS and UTRs are contained within exons, and the promoter of one gene can fall inside an intron or exon of a neighboring gene, including genes on the opposite strand. To assign each base a single feature, the features are combined into one non-overlapping annotation:

1. All features are split into non-overlapping segments, **ignoring strand**, because peaks from ChIP-seq or CUT&Tag are unstranded.
2. Each segment is assigned the highest-priority feature among those covering it:

   | Priority | Class |
   |---|---|
   | 1 | Promoter |
   | 2 | 5' UTR |
   | 3 | 3' UTR |
   | 4 | Exon |
   | 5 | Intron |

3. Adjacent segments with the same class are merged.

The final annotation contains a single `class` column with five categories. CDS is not a separate class: after the UTRs take priority, the remaining `Exon` regions correspond to coding sequence and to exons of non-coding transcripts (e.g. lncRNAs). Regions not covered by any class are intergenic and are reported as `other` during annotation.

In [14]:
GENCODE_PRIORITY <- c("Promoter", "5' UTR", "3' UTR", "Exon", "Intron")

gtf_make_nonoverlap <- function(gr, priority = GENCODE_PRIORITY) {
  feat <- unstrand(granges(gr[gr$type %in% priority]))
  feat$class <- gr$type[gr$type %in% priority]
  cat(sprintf("  Features used: %d\n", length(feat)))
  cat("  "); check_overlap_self(feat)

  # Split into segments, assign the highest-priority category to each segment
  pieces <- disjoin(feat, with.revmap = TRUE)
  pr <- match(feat$class, priority)
  best <- min(extractList(pr, mcols(pieces)$revmap))
  cat(sprintf("  Disjoin: %d features -> %d pieces\n", length(feat), length(pieces)))

  # Merge adjacent segments of the same category
  pieces <- granges(pieces)
  out <- unlist(reduce(split(pieces, priority[best])))
  out$class <- names(out)
  names(out) <- NULL
  out <- sort(out)

  cat(sprintf("  After merging: %d regions\n", length(out)))
  cat("  "); check_overlap_self(out)

  class_tab <- table(factor(out$class, levels = priority))
  print(cbind(n = class_tab, pct = round(100 * prop.table(class_tab), 1)))
  out
}

In [15]:
refs <- c("hg38", "mm10")

for (ref in refs) {
  cat(sprintf("\n==================== %s ====================\n", ref))
  gr <- readRDS(sprintf("GENCODE_rep_%s.rds", ref))

  cat("\n== Extract promoter, intron & UTR ==\n")
  gr <- gtf_add_regions(gr)

  cat("\n== Deal with overlap ==\n")
  gr <- gtf_make_nonoverlap(gr)
  GenomeInfoDb::genome(gr) <- ref

  out_path <- sprintf("GENCODE_genic_%s.rds", ref)
  saveRDS(gr, file = out_path, compress = "xz")
  cat(sprintf("\nSaved: %s\n", out_path))
}


==================== hg38 ====================

== Extract promoter, intron & UTR ==
  Added: 66413 promoters, 259954 introns, 29913 5' UTRs, 23124 3' UTRs
                n  pct
Exon       326367 31.7
Intron     259954 25.2
CDS        192199 18.6
gene        66413  6.4
Promoter    66413  6.4
transcript  66413  6.4
5' UTR      29913  2.9
3' UTR      23124  2.2

== Deal with overlap ==
  Features used: 705771
  Found 669926 internal overlapping interval pairs
  Disjoin: 705771 features -> 703293 pieces
  After merging: 633231 regions
  No internal overlaps found
              n  pct
Promoter  57977  9.2
5' UTR    24120  3.8
3' UTR    21587  3.4
Exon     275897 43.6
Intron   253650 40.1

Saved: GENCODE_genic_hg38.rds

==================== mm10 ====================

== Extract promoter, intron & UTR ==
  Added: 42409 promoters, 208594 introns, 32074 5' UTRs, 23332 3' UTRs
                n  pct
Exon       251004 29.9
Intron     208594 24.9
CDS        196604 23.4
gene        42410  5.1
tr

## knownGene Genic Annotation

As an alternative to GENCODE, the genic annotation can also be built from the UCSC knownGene models distributed as Bioconductor TxDb packages. These are the same models used by default in tools such as ChIPseeker, so the results are directly comparable with analyses based on those tools.

Unlike the GENCODE annotation, all annotated transcripts are used. The TxDb packages do not contain the gene biotypes or transcript evidence tags (MANE, APPRIS, TSL) needed for gene-level filtering and representative transcript selection. The resulting annotation is therefore more comprehensive but includes features from minor isoforms.

#### Step 1: Load TxDb & Filter Chromosomes

The knownGene TxDb package is loaded for each genome (`TxDb.Hsapiens.UCSC.hg38.knownGene`, `TxDb.Mmusculus.UCSC.mm10.knownGene`). Only standard chromosomes are kept, and the mitochondrial chromosome is removed. The package version is recorded, since knownGene content changes between Bioconductor releases.


In [16]:
kg_load_txdb <- function(txdb_pkg) {
  txdb <- getExportedValue(txdb_pkg, txdb_pkg)
  cat(sprintf("  TxDb: %s (version %s)\n", txdb_pkg, as.character(packageVersion(txdb_pkg))))

  # standard chromosomes (chr1-22/19, chrX, chrY), no chrM / random / Un / alt
  all_chr <- seqlevels(txdb)
  chr_list <- grep("^chr([0-9]+|X|Y)$", all_chr, value = TRUE)
  txdb <- GenomeInfoDb::keepSeqlevels(txdb, chr_list, pruning.mode = "coarse")
  cat(sprintf("  Chromosomes: %d -> %d (standard, no chrM)\n", length(all_chr), length(chr_list)))
  txdb
}

#### Step 2: Extract Promoter, UTR, Exon & Intron

Gene structure features are extracted from all transcripts:

- **Promoter**: the 1 kb immediately upstream of each transcript's TSS, excluding the TSS itself and defined according to strand. Regions extending beyond chromosome ends are trimmed.
- **5' UTR / 3' UTR**: taken directly from the TxDb (`fiveUTRsByTranscript`, `threeUTRsByTranscript`), which already distinguishes the two.
- **Exon**: all exons (`exons`).
- **Intron**: gaps between consecutive exons of each transcript (`intronsByTranscript`).

In [17]:
kg_extract_features <- function(txdb, promoter_upstream = 1000L) {
  tx <- transcripts(txdb)
  cat(sprintf("  Transcripts: %d\n", length(tx)))

  prom <- trim(promoters(tx, upstream = promoter_upstream, downstream = 0))
  prom <- prom[width(prom) > 0]

  gr <- c(
    GRanges(prom,                                                   type = "Promoter"),
    GRanges(unlist(fiveUTRsByTranscript(txdb),  use.names = FALSE), type = "5' UTR"),
    GRanges(unlist(threeUTRsByTranscript(txdb), use.names = FALSE), type = "3' UTR"),
    GRanges(exons(txdb),                                            type = "Exon"),
    GRanges(unlist(intronsByTranscript(txdb),   use.names = FALSE), type = "Intron")
  )

  type_tab <- sort(table(gr$type), decreasing = TRUE)
  print(cbind(n = type_tab, pct = round(100 * prop.table(type_tab), 1)))
  gr
}


#### Step 3: Deal with Overlap

Overlaps are resolved exactly as in Step 5 of the GENCODE genic annotation. Features are split into non-overlapping segments ignoring strand, and each segment is assigned the highest-priority class (Promoter > 5' UTR > 3' UTR > Exon > Intron). Adjacent segments with the same class are then merged. The output contains a single `class` column with the same five categories as the GENCODE version.

In [18]:
txdb_pkgs <- c(
  hg38 = "TxDb.Hsapiens.UCSC.hg38.knownGene",
  mm10 = "TxDb.Mmusculus.UCSC.mm10.knownGene"
)

for (ref in names(txdb_pkgs)) {
  cat(sprintf("\n==================== %s ====================\n", ref))

  cat("\n== Load TxDb & filter chromosomes ==\n")
  txdb <- kg_load_txdb(txdb_pkgs[[ref]])

  cat("\n== Extract promoter, UTR, exon & intron ==\n")
  gr <- kg_extract_features(txdb)

  cat("\n== Deal with overlap ==\n")
  gr <- gtf_make_nonoverlap(gr)
  GenomeInfoDb::genome(gr) <- ref

  out_path <- sprintf("knownGene_genic_%s.rds", ref)
  saveRDS(gr, file = out_path, compress = "xz")
  cat(sprintf("\nSaved: %s\n", out_path))
}


==================== hg38 ====================

== Load TxDb & filter chromosomes ==
  TxDb: TxDb.Hsapiens.UCSC.hg38.knownGene (version 3.18.0)
  Chromosomes: 711 -> 24 (standard, no chrM)

== Extract promoter, UTR, exon & intron ==
  Transcripts: 252798
               n  pct
Intron   1396641 51.8
Exon      668271 24.8
Promoter  252798  9.4
3' UTR    207614  7.7
5' UTR    173434  6.4

== Deal with overlap ==
  Features used: 2698758
  Found 48360160 internal overlapping interval pairs
  Disjoin: 2698758 features -> 1304055 pieces
  After merging: 722569 regions
  No internal overlaps found
              n  pct
Promoter 100571 13.9
5' UTR    39067  5.4
3' UTR    72325 10.0
Exon     252596 35.0
Intron   258010 35.7

Saved: knownGene_genic_hg38.rds

==================== mm10 ====================

== Load TxDb & filter chromosomes ==
  TxDb: TxDb.Mmusculus.UCSC.mm10.knownGene (version 3.10.0)
  Chromosomes: 66 -> 21 (standard, no chrM)

== Extract promoter, UTR, exon & intron ==
  Transcr